# 64 Batch mesh statistics

Runs the anonymization pipeline on every valid thesis subject and collects the mesh-level quantities that Chapter 4 actually needs: vertex and face counts before and after deletion, the percentage of vertices removed, the cap-boundary height, and the degenerate-face percentage on the anonymized mesh.

Structural checks (vertex-count delta, mesh validity, degenerate-face ratio, protected-point preservation) come from the shipped `validate_anonymization` -- the same function the production pipeline could invoke -- so the thesis numbers are produced by the function the codebase actually exports, not a notebook re-implementation.

Output: `thesis_results_out/batch_validation.csv`, which populates the mesh-statistics table and the mesh-integrity prose of Chapter 4.

**Prerequisite.** Each subject needs a `Subject{N}_anon_landmarks.tsv` sidecar, written by notebook 48. Subjects without the sidecar are skipped with a warning.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_data import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report,
)
from _thesis_pipeline import run_pipeline

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

from cedalion.geometry.photogrammetry.anonymization import (

    validate_anonymization,

)

## 1. Which subjects are ready?

In [ ]:
ready = available_subjects()

print(f'Ready: {ready}')

missing = missing_report()

if missing:

    print('Missing files:')

    for n, items in missing.items():

        print(f'  Subject{n}: {items}')

## 2. Run pipeline per subject, collect mesh statistics

For each ready subject we load the raw scan and landmarks, run the pipeline (via the wrapper that exposes intermediates) and pass the CTF-aligned head, the anonymized mesh, the deletion mask, and the landmarks to `validate_anonymization`. The wrapper hands us `pct_vertices_removed` and `cap_z_mm` directly; everything else comes from the validator.

In [ ]:
rows = []

for n in ready:

    print(f'--- Subject{n} ---')

    surface_raw = load_raw(n)

    landmarks_raw = load_landmarks(n)

    art = run_pipeline(surface_raw, landmarks_raw, subject=n)



    result = validate_anonymization(

        original_surface=art.surface_ctf,

        anonymized_surface=art.surface_anon_ctf,

        facial_mask=art.mask,

        protected_points=art.landmarks_ctf,

    )



    n_head = art.surface_ctf.nvertices

    pct_removed = 100.0 * result.actual_vertices_removed / max(1, n_head)



    row = {

        'subject': n,

        'n_vertices_raw': surface_raw.nvertices,

        'n_faces_raw': surface_raw.nfaces,

        'n_vertices_head': n_head,

        'n_faces_head': art.surface_ctf.nfaces,

        'mask_size': result.expected_vertices_removed,

        'n_vertices_anonymized': art.surface_anon_ctf.nvertices,

        'n_faces_anonymized': art.surface_anon_ctf.nfaces,

        'vertices_removed': result.actual_vertices_removed,

        'faces_removed': int(art.surface_ctf.nfaces - art.surface_anon_ctf.nfaces),

        'pct_vertices_removed': pct_removed,

        'degenerate_face_pct': result.degenerate_face_pct,

        'protected_point_max_delta_mm': result.protected_point_max_delta_mm,

        'cap_z_mm': art.cap_z,

        'passed': result.passed,

    }

    rows.append(row)

    print(

        f'  head: {n_head:,} v / {art.surface_ctf.nfaces:,} f  ->  '

        f'anon: {art.surface_anon_ctf.nvertices:,} v / {art.surface_anon_ctf.nfaces:,} f  '

        f'(-{pct_removed:.1f}%, degen {result.degenerate_face_pct:.3f}%, '

        f'cap_z {art.cap_z:.1f} mm)  {result.summary}'

    )

## 3. Summary table

One row per subject. Columns feed the mesh-statistics table and the mesh-integrity prose of Chapter 4.

In [ ]:
df = pd.DataFrame(rows)

if len(df):

    df = df.sort_values('subject').reset_index(drop=True)

df

## 4. Save CSV

In [ ]:
out = OUT_DIR / 'batch_validation.csv'

df.to_csv(out, index=False)

print(f'Wrote {out} ({len(df)} rows)')